In [2]:
# ============================================================
# CELL 1 - Installations and Imports
# Run this cell first every time you open this notebook
# ============================================================

# Install all required libraries
!pip install openai pybioportal pandas python-docx lifelines matplotlib scipy -q

# ── Imports ──────────────────────────────────────────────────

# OpenAI - to talk to GPT
from openai import OpenAI

# pyBioPortal - to access cBioPortal cancer genomics data
import pybioportal

# Pandas - for handling data as tables
import pandas as pd

# python-docx - for creating Word documents
from docx import Document
from docx.shared import Pt

# Survival analysis - KM curves, Cox regression, log rank tests
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

# Plotting - for KM curves and visualizations
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # saves plots as files, no popups

# Statistical tests - p values, confidence intervals
from scipy import stats
import scipy.stats as sp

# Built in Python libraries
import json
import getpass
from datetime import datetime
import os
import warnings
warnings.filterwarnings('ignore')  # keeps output clean

print("✅ All libraries installed and imported successfully!")
print("\nLibraries ready:")
print("→ OpenAI         - GPT analysis")
print("→ pyBioPortal    - cBioPortal cancer data")
print("→ Pandas         - data manipulation")
print("→ Lifelines      - survival analysis and KM curves")
print("→ Matplotlib     - plotting and visualizations")
print("→ Scipy          - statistical tests")
print("→ python-docx    - Word document export")

  You can safely remove it manually.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


✅ All libraries installed and imported successfully!

Libraries ready:
→ OpenAI         - GPT analysis
→ pyBioPortal    - cBioPortal cancer data
→ Pandas         - data manipulation
→ Lifelines      - survival analysis and KM curves
→ Matplotlib     - plotting and visualizations
→ Scipy          - statistical tests
→ python-docx    - Word document export


In [13]:
# ============================================================
# ESR1 VALIDATION AGENT
# Validating Flatiron ESR1 Mutation Findings using
# MSK Metastatic Breast Cancer cBioPortal Dataset
# ============================================================
#
# STUDY BACKGROUND:
# A Flatiron Health retrospective study analyzed ESR1 mutation
# testing and positivity rates in ER+/HER2- metastatic breast
# cancer patients who initiated 1L therapy with aromatase
# inhibitors or SERDs ± CDK4/6 inhibitors (Jan 2020-Dec 2024)
#
# KEY FLATIRON FINDINGS TO VALIDATE:
# → ESR1 positivity at 1L initiation: 25% among tested patients
# → ESR1 positivity at 2L initiation: 32-34% among tested patients
# → Only 49% of patients underwent ESR1 testing
# → Mean 1.6 tests per patient
# → 60% tissue-only, 40% blood-only testing
#
# VALIDATION APPROACH:
# We use the MSK Metastatic Breast Cancer dataset
# (Cancer Discovery 2022, n=1116 patients) from cBioPortal
# to independently validate ESR1 mutation positivity rates
# cBioPortal is a public cancer genomics database with
# real world genomic and clinical data from MSK patients
#
# DATA STRUCTURE — THREE LINKED TABLES:
# (Same concept as relational databases in Flatiron/Optum)
#
# TABLE 1 → Patient Table (one row per patient)
#            Demographics: age, sex, race, ethnicity
#            Key: patientId
#
# TABLE 2 → Sample Table (one row per tumor sample)
#            Tumor characteristics: cancer type,
#            metastatic site, tumor mutation burden
#            Keys: sampleId + patientId
#            Links to Table 1 via patientId
#            Note: one patient can have multiple samples
#
# TABLE 3 → Mutations Table (one row per mutation)
#            Genomic data: ESR1 mutation type,
#            protein change, mutation status
#            Keys: sampleId + patientId
#            Links to Table 1 and 2 via patientId
#            Note: one patient can have multiple mutations
#
# ANALYSIS PLAN:
# Step 1 → Fetch patient demographics (Table 1)
# Step 2 → Fetch sample clinical data (Table 2)
# Step 3 → Fetch ESR1 mutation data (Table 3)
# Step 4 → Calculate ESR1 positivity rate
# Step 5 → Compare against Flatiron findings
# Step 6 → Survival analysis (KM curves) by ESR1 status
# Step 7 → Export full validation report to Word document
#
# LIBRARIES USED:
# pyBioPortal  → connect to cBioPortal API
# pandas       → data manipulation and table merging
# lifelines    → survival analysis and KM curves
# matplotlib   → plotting KM curves
# scipy        → statistical tests and p-values
# openai       → GPT interpretation of findings
# python-docx  → Word document export
#
# VALIDATION HYPOTHESIS:
# If Flatiron findings are generalizable, we expect
# ESR1 positivity rates in cBioPortal MSK data to be
# within 5 percentage points of Flatiron findings (25%)
# ============================================================

In [3]:
# ============================================================
# CELL 2 - API Key
# Only OpenAI key needed
# cBioPortal is completely free - no key required
# ============================================================

# OpenAI API key - for GPT analysis and interpretation
openai_key = getpass.getpass("Paste your OpenAI API key: ")

# Create OpenAI client
# This is our connection to GPT
client = OpenAI(api_key=openai_key)

print("✅ OpenAI client ready!")
print("✅ cBioPortal is free - no key needed!")

✅ OpenAI client ready!
✅ cBioPortal is free - no key needed!


In [6]:
# ============================================================
# CELL 3 - Find the correct study ID from cBioPortal
# ============================================================
# Before fetching data we need the exact study ID
# This cell searches all available studies
# and finds the right one for us automatically
# ============================================================

# Import the studies module from pyBioPortal
from pybioportal import studies as st
from pybioportal import clinical_data as cd

print("Searching cBioPortal for all breast cancer studies...")

# Fetch all available studies from cBioPortal
all_studies = st.get_all_studies()

# Filter for studies that contain "breast" and "msk"
# str.contains() searches within text columns
# case=False means ignore upper/lower case
breast_msk_studies = all_studies[
    all_studies['studyId'].str.contains('breast', case=False) &
    all_studies['studyId'].str.contains('msk', case=False)
]

print(f"\nBreast cancer MSK studies found:")
print(breast_msk_studies[['studyId', 'name', 'allSampleCount']])

Searching cBioPortal for all breast cancer studies...

Breast cancer MSK studies found:
                   studyId                                               name  \
294        breast_msk_2018              Breast Cancer (MSK, Cancer Cell 2018)   
347        breast_msk_2025                Breast Cancer (MSK, Nat Genet 2025)   
420   breast_ink4_msk_2021  Metastatic Breast Cancer (MSK, Cancer Discover...   
421  breast_msk_cfdna_2026                    Breast Cancer ctDNA (MSK, 2026)   

     allSampleCount  
294               1  
347               1  
420               1  
421               1  


In [7]:
# ============================================================
# CELL 4 - Fetch Patient Clinical Data from cBioPortal
# ============================================================
# What this cell does:
# 1. Uses the correct study ID we found in Cell 3
# 2. Fetches all patient clinical data from MSK MBC study
# 3. Returns data as a table - one row per patient
# 4. Shows us what columns/attributes are available
#
# Why we do this first:
# Before writing any analysis code we need to understand
# what data is actually available — exact column names,
# formats, and what attributes exist
# You cannot filter for ER+/HER2- without knowing
# what those columns are called in this dataset
# Same concept as proc contents + proc print in SAS
#
# Why this study ID?
# breast_ink4_msk_2021 is the correct ID for
# Metastatic Breast Cancer (MSK, Cancer Discovery 2022)
# We found this by searching all studies in Cell 3
# The ID has 2021 but the publication is from 2022
# ============================================================

# Correct study ID found from our search in Cell 3
# breast_ink4_msk_2021 = MSK Cancer Discovery 2022 study
# This is the study with 1365 samples we saw on cBioPortal
STUDY_ID = "breast_ink4_msk_2021"

print(f"Connecting to cBioPortal...")
print(f"Fetching study: {STUDY_ID}")
print(f"Please wait — this may take 30-60 seconds...")

# Fetch all patient level clinical data for this study
# study_id           → which study to fetch
# clinical_data_type → PATIENT means one row per person
#                      not one row per tissue sample
#                      remember: one patient can have
#                      multiple samples taken at different
#                      time points or tumor sites
# ret_format=WIDE    → each column is one clinical attribute
#                      like an Excel spreadsheet
#                      much easier to filter and analyze
#                      default format has multiple rows
#                      per patient which is harder to work with
clinical_data = cd.fetch_all_clinical_data_in_study(
    study_id=STUDY_ID,
    clinical_data_type="PATIENT",
    ret_format="WIDE"
)

# Show what came back
# len() counts number of rows = number of patients
print(f"\n✅ Data fetched successfully!")
print(f"Total patients: {len(clinical_data)}")

# head(3) shows first 3 rows of the table
# so we can see what the data looks like
print(f"\nFirst 3 rows:")
print(clinical_data.head(3))

# columns.tolist() converts column names to a list
# so we can see ALL available attributes
# This tells us what we can filter and analyze
# We need to find:
# → ER status column name
# → HER2 status column name
# → Survival column name
# → ESR1 mutation column name
print(f"\nAll columns available:")
print(clinical_data.columns.tolist())

Connecting to cBioPortal...
Fetching study: breast_ink4_msk_2021
Please wait — this may take 30-60 seconds...

✅ Data fetched successfully!
Total patients: 1116

First 3 rows:
clinicalAttributeId                          uniquePatientKey  patientId  \
0                    UC0wMDA0MDU4OmJyZWFzdF9pbms0X21za18yMDIx  P-0004058   
1                    UC0wMDA0MDUyOmJyZWFzdF9pbms0X21za18yMDIx  P-0004052   
2                    UC0wMDA0MTI3OmJyZWFzdF9pbms0X21za18yMDIx  P-0004127   

clinicalAttributeId               studyId AGE_CURRENT  \
0                    breast_ink4_msk_2021          61   
1                    breast_ink4_msk_2021          81   
2                    breast_ink4_msk_2021          58   

clinicalAttributeId                       ETHNICITY                  RACE  \
0                         Non-Spanish; Non-Hispanic  PT REFUSED TO ANSWER   
1                    Unknown whether Spanish or not                 WHITE   
2                         Non-Spanish; Non-Hispanic        

In [8]:
# ============================================================
# CELL 5 - Fetch Sample Level Clinical Data
# ============================================================
# Patient level data gave us demographics only
# Sample level data gives us tumor specific attributes
# This is where ER status, HER2 status and 
# survival data will be found
# Same concept as having a patient table and
# a separate tumor/sample table in your Flatiron studies
# ============================================================

print(f"Fetching sample level clinical data...")
print(f"This contains tumor specific attributes...")
print(f"ER status, HER2 status, survival data etc.")

# Fetch sample level clinical data
# clinical_data_type="SAMPLE" instead of "PATIENT"
# Everything else stays the same
sample_data = cd.fetch_all_clinical_data_in_study(
    study_id=STUDY_ID,
    clinical_data_type="SAMPLE",
    ret_format="WIDE"
)

print(f"\n✅ Sample data fetched successfully!")
print(f"Total samples: {len(sample_data)}")
print(f"\nFirst 3 rows:")
print(sample_data.head(3))
print(f"\nAll columns available in sample data:")
print(sample_data.columns.tolist())

Fetching sample level clinical data...
This contains tumor specific attributes...
ER status, HER2 status, survival data etc.

✅ Sample data fetched successfully!
Total samples: 1365

First 3 rows:
clinicalAttributeId                                    uniqueSampleKey  \
0                    UC0wMDA0MDU4LVQwMS1JTTU6YnJlYXN0X2luazRfbXNrXz...   
1                    UC0wMDA0MDUyLVQwMS1JTTU6YnJlYXN0X2luazRfbXNrXz...   
2                    UC0wMDA0MTI3LVQwMS1JTTU6YnJlYXN0X2luazRfbXNrXz...   

clinicalAttributeId                          uniquePatientKey  \
0                    UC0wMDA0MDU4OmJyZWFzdF9pbms0X21za18yMDIx   
1                    UC0wMDA0MDUyOmJyZWFzdF9pbms0X21za18yMDIx   
2                    UC0wMDA0MTI3OmJyZWFzdF9pbms0X21za18yMDIx   

clinicalAttributeId           sampleId  patientId               studyId  \
0                    P-0004058-T01-IM5  P-0004058  breast_ink4_msk_2021   
1                    P-0004052-T01-IM5  P-0004052  breast_ink4_msk_2021   
2                   

In [10]:
# ============================================================
# CELL 6 - Find correct function names in mutations module
# ============================================================
# Before calling any function we check what functions
# are actually available in the mutations module
# This prevents the AttributeError we just saw
# Same as reading the manual before using a new tool
# ============================================================

# Import mutations module
from pybioportal import mutations as mut

# List all available functions in the mutations module
# dir() returns all attributes and functions of an object
# We filter for functions that don't start with _ 
# (those are internal functions we don't need)
available_functions = [f for f in dir(mut) if not f.startswith('_')]

print("Available functions in pyBioPortal mutations module:")
for func in available_functions:
    print(f"  → {func}")

Available functions in pyBioPortal mutations module:
  → base_url
  → fetch_muts_in_mol_prof
  → fetch_muts_in_multiple_mol_profs
  → get_muts_in_mol_prof_by_sample_list_id
  → process_response
  → requests


In [12]:
# ============================================================
# CELL 7 - Fetch ESR1 Mutation Data
# ============================================================
# Now we know the correct function name from Cell 6
# get_muts_in_mol_prof_by_sample_list_id
#
# What this cell does:
# 1. Connects to the mutations module
# 2. Fetches all ESR1 mutations in the MSK study
# 3. Returns mutation type, patient ID, sample ID
#
# Key concepts:
# molecular_profile_id → the specific genomic dataset
#                        in cBioPortal every study has
#                        a mutations profile named:
#                        studyId + "_mutations"
#
# sample_list_id       → which samples to include
#                        studyId + "_all" means all samples
#
# entrez_gene_id       → ESR1's unique gene ID = 2099
#                        every gene has a unique number
#                        in biological databases
# ============================================================

print("Fetching ESR1 mutation data from MSK study...")
print("Please wait...")

# Fetch all ESR1 mutations in this study
# molecular_profile_id → studyId + "_mutations"
# sample_list_id       → studyId + "_all" = all samples
# entrez_gene_id       → 2099 is ESR1's unique gene ID
esr1_mutations = mut.get_muts_in_mol_prof_by_sample_list_id(
    molecular_profile_id=f"{STUDY_ID}_mutations",
    sample_list_id=f"{STUDY_ID}_all",
    entrez_gene_id=2099
)

print(f"\n✅ ESR1 mutation data fetched!")
print(f"Total ESR1 mutations found: {len(esr1_mutations)}")
print(f"\nFirst 3 rows:")
print(esr1_mutations.head(3))
print(f"\nAll columns available:")
print(esr1_mutations.columns.tolist())

Fetching ESR1 mutation data from MSK study...
Please wait...

✅ ESR1 mutation data fetched!
Total ESR1 mutations found: 305

First 3 rows:
                                     uniqueSampleKey  \
0  UC0wMDAwMDE1LVQwMS1JTTM6YnJlYXN0X2luazRfbXNrXz...   
1  UC0wMDAwMDY2LVQwMS1JTTM6YnJlYXN0X2luazRfbXNrXz...   
2  UC0wMDAwMDY2LVQwMS1JTTM6YnJlYXN0X2luazRfbXNrXz...   

                           uniquePatientKey              molecularProfileId  \
0  UC0wMDAwMDE1OmJyZWFzdF9pbms0X21za18yMDIx  breast_ink4_msk_2021_mutations   
1  UC0wMDAwMDY2OmJyZWFzdF9pbms0X21za18yMDIx  breast_ink4_msk_2021_mutations   
2  UC0wMDAwMDY2OmJyZWFzdF9pbms0X21za18yMDIx  breast_ink4_msk_2021_mutations   

            sampleId  patientId  entrezGeneId               studyId center  \
0  P-0000015-T01-IM3  P-0000015          2099  breast_ink4_msk_2021  MSKCC   
1  P-0000066-T01-IM3  P-0000066          2099  breast_ink4_msk_2021  MSKCC   
2  P-0000066-T01-IM3  P-0000066          2099  breast_ink4_msk_2021  MSKCC   

  muta

In [16]:
# ============================================================
# CELL 8 - Calculate ESR1 Mutation Positivity Rate
# ============================================================
# This is the core calculation for our validation study
# We calculate what % of patients have ESR1 mutations
# in the MSK cBioPortal dataset
#
# HOW WE CALCULATE:
# 1. Total patients → from Cell 4 (1116 patients)
# 2. Patients with ESR1 → count unique patientIds
#    in the mutations table from Cell 7
# 3. Positivity rate → (ESR1 patients / total) x 100
#
# WHY UNIQUE PATIENTS?
# One patient can have multiple ESR1 mutations
# For example P-0000066 had both L536H and D538G
# We count them as ONE patient with ESR1 mutation
# Same approach used in Flatiron study
# Same as counting distinct patients in SAS:
# proc sql;
#   select count(distinct patientId)
#   from mutations
#   where gene = 'ESR1';
# quit;
#
# NOTE ON COMPARISON:
# We do NOT compare against any hardcoded value here
# Comparison against published literature will be
# done dynamically by the RWE Research Agent
# which will search PubMed for published ESR1 rates
# and compare them against our cBioPortal findings
# This makes the system reusable for any gene or study
# ============================================================

# Total patients in the study
# We fetched this in Cell 4
# len() counts number of rows in the patient table
total_patients = len(clinical_data)

# Count unique patients with at least one ESR1 mutation
# nunique() counts unique values — each patient once
# even if they have multiple ESR1 mutations
patients_with_esr1 = esr1_mutations['patientId'].nunique()

# Patients WITHOUT ESR1 mutations
patients_without_esr1 = total_patients - patients_with_esr1

# Calculate positivity rate
# round() to 1 decimal also fixes floating point issues
# like 3.6999999999999993 becoming 3.7
esr1_positivity_rate = round(
    (patients_with_esr1 / total_patients) * 100, 1
)

# ── PRINT RESULTS ─────────────────────────────────────────
print("=" * 55)
print("ESR1 MUTATION POSITIVITY ANALYSIS")
print("MSK Metastatic Breast Cancer (Cancer Discovery 2022)")
print("=" * 55)
print(f"Total patients in study:           {total_patients}")
print(f"Patients WITH ESR1 mutations:      {patients_with_esr1} ({esr1_positivity_rate}%)")
print(f"Patients WITHOUT ESR1 mutations:   {patients_without_esr1} ({round(100 - esr1_positivity_rate, 1)}%)")
print(f"Total ESR1 mutation events:        {len(esr1_mutations)}")

# ── STORE RESULTS FOR AGENT PIPELINE ─────────────────────
# We store results as a dictionary
# so the comparison agent can easily use them later
# Think of it like a structured output handed off
# to the next agent in the pipeline
esr1_results = {
    "total_patients": total_patients,
    "patients_with_esr1": patients_with_esr1,
    "patients_without_esr1": patients_without_esr1,
    "esr1_positivity_rate": esr1_positivity_rate,
    "total_mutations": len(esr1_mutations),
    "study": "MSK Metastatic Breast Cancer Cancer Discovery 2022",
    "database": "cBioPortal"
}

print(f"\n✅ Results stored for agent pipeline!")
print(f"esr1_results: {esr1_results}")

ESR1 MUTATION POSITIVITY ANALYSIS
MSK Metastatic Breast Cancer (Cancer Discovery 2022)
Total patients in study:           1116
Patients WITH ESR1 mutations:      238 (21.3%)
Patients WITHOUT ESR1 mutations:   878 (78.7%)
Total ESR1 mutation events:        305

✅ Results stored for agent pipeline!
esr1_results: {'total_patients': 1116, 'patients_with_esr1': 238, 'patients_without_esr1': 878, 'esr1_positivity_rate': 21.3, 'total_mutations': 305, 'study': 'MSK Metastatic Breast Cancer Cancer Discovery 2022', 'database': 'cBioPortal'}


In [17]:
# ============================================================
# CELL 9a - Merge All Three Tables
# ============================================================
# Before building Table 1 we need to combine our
# three separate tables into one complete patient table
#
# THREE TABLES WE HAVE:
# clinical_data   → patient demographics (1116 rows)
# sample_data     → tumor characteristics (1365 rows)
# esr1_mutations  → ESR1 mutation data (305 rows)
#
# MERGING STRATEGY:
# Step 1 → Merge clinical_data + sample_data
#           on patientId
#           Use one sample per patient
#           (some patients have multiple samples)
#           We take the FIRST sample per patient
#
# Step 2 → Add ESR1 status column
#           If patientId appears in esr1_mutations
#           → ESR1_POSITIVE = Yes
#           If patientId does NOT appear
#           → ESR1_POSITIVE = No
#
# This is exactly like a LEFT JOIN in SAS:
# proc sql;
#   create table merged as
#   select a.*, b.METASTATIC_SITE, b.MSI_TYPE,
#          b.TMB_NONSYNONYMOUS
#   from clinical_data a
#   left join sample_data b
#   on a.patientId = b.patientId;
# quit;
# ============================================================

print("Merging patient, sample and mutation tables...")

# ── STEP 1: Handle multiple samples per patient ───────────
# Some patients have more than one sample
# We keep only the FIRST sample per patient
# to avoid counting the same patient twice
# Same as deduplication in SAS:
# proc sort data=sample_data nodupkey;
#   by patientId;
# run;
sample_data_dedup = sample_data.drop_duplicates(
    subset='patientId',  # keep first occurrence per patient
    keep='first'         # keep first sample if multiple exist
)

print(f"Samples after deduplication: {len(sample_data_dedup)}")
print(f"(Started with {len(sample_data)} samples)")

# ── STEP 2: Merge clinical + sample data ─────────────────
# LEFT JOIN — keep all patients from clinical_data
# even if they have no matching sample data
# merge() is pandas equivalent of SQL LEFT JOIN
# on='patientId' means match rows where patientId matches
merged_data = clinical_data.merge(
    sample_data_dedup[[
        'patientId',
        'METASTATIC_SITE',
        'MSI_TYPE',
        'TMB_NONSYNONYMOUS',
        'CANCER_TYPE_DETAILED',
        'SAMPLE_TYPE',
        'MUTATION_COUNT'
    ]],
    on='patientId',
    how='left'   # keep all patients even without sample data
)

print(f"\nAfter merging clinical + sample: {len(merged_data)} patients")

# ── STEP 3: Add ESR1 status column ───────────────────────
# Create a list of patientIds who have ESR1 mutations
# Then check if each patient in merged_data is in that list
# isin() checks if value exists in a list
# like an IN operator in SAS:
# if patientId in esr1_positive_patients then ESR1_STATUS='Positive'
esr1_positive_patients = esr1_mutations['patientId'].unique()

# Create ESR1_STATUS column
# If patientId is in esr1_positive_patients → Positive
# Otherwise → Negative
merged_data['ESR1_STATUS'] = merged_data['patientId'].apply(
    lambda x: 'ESR1 Positive' if x in esr1_positive_patients 
              else 'ESR1 Negative'
)

# ── VERIFY ────────────────────────────────────────────────
print(f"\n✅ Final merged table: {len(merged_data)} patients")
print(f"\nESR1 Status distribution:")
print(merged_data['ESR1_STATUS'].value_counts())
print(f"\nColumns in merged table:")
print(merged_data.columns.tolist())

Merging patient, sample and mutation tables...
Samples after deduplication: 1116
(Started with 1365 samples)

After merging clinical + sample: 1116 patients

✅ Final merged table: 1116 patients

ESR1 Status distribution:
ESR1_STATUS
ESR1 Negative    878
ESR1 Positive    238
Name: count, dtype: int64

Columns in merged table:
['uniquePatientKey', 'patientId', 'studyId', 'AGE_CURRENT', 'ETHNICITY', 'RACE', 'SAMPLE_COUNT', 'SEX', 'METASTATIC_SITE', 'MSI_TYPE', 'TMB_NONSYNONYMOUS', 'CANCER_TYPE_DETAILED', 'SAMPLE_TYPE', 'MUTATION_COUNT', 'ESR1_STATUS']


In [18]:
# ============================================================
# CELL 9b - Quick Data Check Before Table 1
# ============================================================
# Before building Table 1 we need to see
# what unique values exist in each column
# So we know how to group and present them
# Same as proc freq in SAS before building
# a summary table
# ============================================================

# Check each column we plan to include in Table 1
columns_to_check = [
    'AGE_CURRENT',
    'SEX', 
    'RACE',
    'ETHNICITY',
    'METASTATIC_SITE',
    'MSI_TYPE',
    'CANCER_TYPE_DETAILED'
]

for col in columns_to_check:
    print(f"\n── {col} ──")
    print(merged_data[col].value_counts().head(10))


── AGE_CURRENT ──
AGE_CURRENT
58    46
56    38
60    37
67    37
59    36
68    36
63    36
62    36
57    35
73    34
Name: count, dtype: int64

── SEX ──
SEX
Female    1093
Male        14
Name: count, dtype: int64

── RACE ──
RACE
WHITE                            878
BLACK OR AFRICAN AMERICAN         87
ASIAN-FAR EAST/INDIAN SUBCONT     54
PT REFUSED TO ANSWER              49
OTHER                             28
UNKNOWN                            7
NO VALUE ENTERED                   2
NATIVE AMERICAN-AM IND/ALASKA      1
Name: count, dtype: int64

── ETHNICITY ──
ETHNICITY
Non-Spanish; Non-Hispanic                             973
Unknown whether Spanish or not                         58
Spanish  NOS; Hispanic NOS, Latino NOS                 53
South/Central America (except Brazil)                   8
Cuban                                                   4
Puerto Rican                                            4
Dominican Republic                                      2
Other Span

In [21]:
# ============================================================
# CELL 9c - Table 1: Patient Demographics and Characteristics
# ============================================================
# Standard Table 1 for RWE studies
# Shows overall population AND stratified by ESR1 status
# This is the first table in any published RWE paper
#
# FORMAT:
# Continuous variables → median (IQR)
#                        e.g. Age: 58 (50-65)
# Categorical variables → n (%)
#                        e.g. Female: 1093 (97.9%)
#
# STRATIFICATION:
# Overall (N=1116) | ESR1+ (N=238) | ESR1- (N=878)
#
# WHY THIS MATTERS:
# If ESR1+ and ESR1- patients look very different
# demographically it means the groups are not
# comparable and we need to adjust for confounding
# Same concept as your IPTW work in Flatiron
# ============================================================

import numpy as np

# ── HELPER FUNCTIONS ──────────────────────────────────────

def median_iqr(series):
    """
    Calculate median and IQR for continuous variables
    Returns formatted string like "58.0 (50.0 - 66.0)"
    """
    # Convert to numeric first
    # errors='coerce' turns non-numeric values to NaN
    numeric = pd.to_numeric(series, errors='coerce')
    
    # Calculate median and quartiles
    median = numeric.median()
    q1 = numeric.quantile(0.25)  # 25th percentile
    q3 = numeric.quantile(0.75)  # 75th percentile
    
    return f"{median:.1f} ({q1:.1f} - {q3:.1f})"

def n_pct(series, value):
    """
    Calculate n and % for categorical variables
    Returns formatted string like "1093 (97.9%)"
    """
    count = (series == value).sum()
    pct = round((count / len(series)) * 100, 1)
    return f"{count} ({pct}%)"

# ── SPLIT INTO THREE GROUPS ───────────────────────────────
# Overall population
overall = merged_data

# ESR1 positive patients only
esr1_pos = merged_data[merged_data['ESR1_STATUS'] == 'ESR1 Positive']

# ESR1 negative patients only
esr1_neg = merged_data[merged_data['ESR1_STATUS'] == 'ESR1 Negative']

# ── GROUP RACE INTO CLEANER CATEGORIES ───────────────────
# Too many small categories — group into main ones
def group_race(race):
    """Groups race into main categories for cleaner table"""
    if race == 'WHITE':
        return 'White'
    elif race == 'BLACK OR AFRICAN AMERICAN':
        return 'Black or African American'
    elif race == 'ASIAN-FAR EAST/INDIAN SUBCONT':
        return 'Asian'
    else:
        return 'Other/Unknown'

merged_data['RACE_GROUPED'] = merged_data['RACE'].apply(group_race)
esr1_pos = merged_data[merged_data['ESR1_STATUS'] == 'ESR1 Positive']
esr1_neg = merged_data[merged_data['ESR1_STATUS'] == 'ESR1 Negative']

# ── GROUP ETHNICITY ───────────────────────────────────────
def group_ethnicity(eth):
    """Groups ethnicity into Hispanic vs Non-Hispanic"""
    if 'Non-Spanish' in str(eth):
        return 'Non-Hispanic'
    elif eth == 'Unknown whether Spanish or not':
        return 'Unknown'
    else:
        return 'Hispanic'

merged_data['ETHNICITY_GROUPED'] = merged_data['ETHNICITY'].apply(
    group_ethnicity
)
esr1_pos = merged_data[merged_data['ESR1_STATUS'] == 'ESR1 Positive']
esr1_neg = merged_data[merged_data['ESR1_STATUS'] == 'ESR1 Negative']

# ── GROUP CANCER TYPE ─────────────────────────────────────
def group_cancer_type(cancer):
    """Groups detailed cancer types into main categories"""
    if 'Ductal' in str(cancer):
        return 'Invasive Ductal'
    elif 'Lobular' in str(cancer):
        return 'Invasive Lobular'
    else:
        return 'Other'

merged_data['CANCER_TYPE_GROUPED'] = merged_data[
    'CANCER_TYPE_DETAILED'
].apply(group_cancer_type)
esr1_pos = merged_data[merged_data['ESR1_STATUS'] == 'ESR1 Positive']
esr1_neg = merged_data[merged_data['ESR1_STATUS'] == 'ESR1 Negative']

# ── BUILD TABLE 1 ─────────────────────────────────────────
print("=" * 75)
print(f"{'TABLE 1: PATIENT CHARACTERISTICS':^75}")
print("=" * 75)
print(f"{'Characteristic':<35} {'Overall':^15} {'ESR1+':^12} {'ESR1-':^12}")
print(f"{'':35} {'(N=1116)':^15} {'(N=238)':^12} {'(N=878)':^12}")
print("-" * 75)

# ── AGE ───────────────────────────────────────────────────
print(f"\n{'Age — median (IQR)':<35}", end="")
print(f"{median_iqr(overall['AGE_CURRENT']):^15}", end="")
print(f"{median_iqr(esr1_pos['AGE_CURRENT']):^12}", end="")
print(f"{median_iqr(esr1_neg['AGE_CURRENT']):^12}")

# ── SEX ───────────────────────────────────────────────────
print(f"\n{'Sex':<35}")
print(f"  {'Female — n (%)' :<33}", end="")
print(f"{n_pct(overall['SEX'], 'Female'):^15}", end="")
print(f"{n_pct(esr1_pos['SEX'], 'Female'):^12}", end="")
print(f"{n_pct(esr1_neg['SEX'], 'Female'):^12}")

print(f"  {'Male — n (%)' :<33}", end="")
print(f"{n_pct(overall['SEX'], 'Male'):^15}", end="")
print(f"{n_pct(esr1_pos['SEX'], 'Male'):^12}", end="")
print(f"{n_pct(esr1_neg['SEX'], 'Male'):^12}")

# ── RACE ──────────────────────────────────────────────────
print(f"\n{'Race':<35}")
for race_cat in ['White', 'Black or African American', 
                  'Asian', 'Other/Unknown']:
    print(f"  {race_cat + ' — n (%)' :<33}", end="")
    print(f"{n_pct(overall['RACE_GROUPED'], race_cat):^15}", end="")
    print(f"{n_pct(esr1_pos['RACE_GROUPED'], race_cat):^12}", end="")
    print(f"{n_pct(esr1_neg['RACE_GROUPED'], race_cat):^12}")

# ── ETHNICITY ─────────────────────────────────────────────
print(f"\n{'Ethnicity':<35}")
for eth_cat in ['Non-Hispanic', 'Hispanic', 'Unknown']:
    print(f"  {eth_cat + ' — n (%)' :<33}", end="")
    print(f"{n_pct(overall['ETHNICITY_GROUPED'], eth_cat):^15}", end="")
    print(f"{n_pct(esr1_pos['ETHNICITY_GROUPED'], eth_cat):^12}", end="")
    print(f"{n_pct(esr1_neg['ETHNICITY_GROUPED'], eth_cat):^12}")

# ── CANCER TYPE ───────────────────────────────────────────
print(f"\n{'Cancer Type':<35}")
for cancer_cat in ['Invasive Ductal', 'Invasive Lobular', 'Other']:
    print(f"  {cancer_cat + ' — n (%)' :<33}", end="")
    print(f"{n_pct(overall['CANCER_TYPE_GROUPED'], cancer_cat):^15}",
          end="")
    print(f"{n_pct(esr1_pos['CANCER_TYPE_GROUPED'], cancer_cat):^12}",
          end="")
    print(f"{n_pct(esr1_neg['CANCER_TYPE_GROUPED'], cancer_cat):^12}")

# ── METASTATIC SITE ───────────────────────────────────────
print(f"\n{'Metastatic Site (top 5)':<35}")
for site in ['Liver', 'Bone', 'Lymph Node', 'Lung', 'Skin']:
    print(f"  {site + ' — n (%)' :<33}", end="")
    print(f"{n_pct(overall['METASTATIC_SITE'], site):^15}", end="")
    print(f"{n_pct(esr1_pos['METASTATIC_SITE'], site):^12}", end="")
    print(f"{n_pct(esr1_neg['METASTATIC_SITE'], site):^12}")

# ── MSI STATUS ────────────────────────────────────────────
print(f"\n{'MSI Status':<35}")
for msi_cat in ['Stable', 'Indeterminate', 'Instable']:
    print(f"  {msi_cat + ' — n (%)' :<33}", end="")
    print(f"{n_pct(overall['MSI_TYPE'], msi_cat):^15}", end="")
    print(f"{n_pct(esr1_pos['MSI_TYPE'], msi_cat):^12}", end="")
    print(f"{n_pct(esr1_neg['MSI_TYPE'], msi_cat):^12}")

# ── TMB ───────────────────────────────────────────────────
print(f"\n{'TMB — median (IQR)':<35}", end="")
print(f"{median_iqr(overall['TMB_NONSYNONYMOUS']):^15}", end="")
print(f"{median_iqr(esr1_pos['TMB_NONSYNONYMOUS']):^12}", end="")
print(f"{median_iqr(esr1_neg['TMB_NONSYNONYMOUS']):^12}")

print("\n" + "=" * 75)
print("IQR = Interquartile Range")
print("ESR1+ = ESR1 mutation positive")
print("ESR1- = ESR1 mutation negative")

# ── STORE TABLE 1 RESULTS ─────────────────────────────────
# Store key demographic findings for agent pipeline
# These will be passed to comparison and report agents
table1_results = {
    "total_patients": len(overall),
    "esr1_positive": len(esr1_pos),
    "esr1_negative": len(esr1_neg),
    "median_age_overall": pd.to_numeric(
        overall['AGE_CURRENT'], errors='coerce'
    ).median(),
    "pct_female": round(
        (overall['SEX'] == 'Female').sum() / len(overall) * 100, 1
    ),
    "pct_white": round(
        (overall['RACE_GROUPED'] == 'White').sum() / len(overall) * 100, 1
    ),
    "top_metastatic_site": overall['METASTATIC_SITE'].value_counts().index[0]
}

print(f"\n✅ Table 1 results stored for agent pipeline!")

                     TABLE 1: PATIENT CHARACTERISTICS                      
Characteristic                          Overall        ESR1+        ESR1-    
                                       (N=1116)       (N=238)      (N=878)   
---------------------------------------------------------------------------

Age — median (IQR)                 61.0 (53.0 - 70.0)62.0 (54.0 - 70.0)61.0 (53.0 - 70.0)

Sex                                
  Female — n (%)                    1093 (97.9%)  236 (99.2%) 857 (97.6%) 
  Male — n (%)                        14 (1.3%)     1 (0.4%)   13 (1.5%)  

Race                               
  White — n (%)                      878 (78.7%)  200 (84.0%) 678 (77.2%) 
  Black or African American — n (%)   87 (7.8%)    17 (7.1%)   70 (8.0%)  
  Asian — n (%)                       54 (4.8%)     6 (2.5%)   48 (5.5%)  
  Other/Unknown — n (%)               97 (8.7%)    15 (6.3%)   82 (9.3%)  

Ethnicity                          
  Non-Hispanic — n (%)               973

In [22]:
# ============================================================
# CELL 10a - Check if Survival Data Exists
# ============================================================
# Before running KM curves we must verify that
# OS_STATUS and OS_MONTHS exist in this dataset
# We cannot run survival analysis without these
#
# OS_STATUS → whether patient is alive or dead
#             usually coded as:
#             "0:LIVING" or "1:DECEASED"
#
# OS_MONTHS → how many months patient survived
#             from diagnosis to last contact/death
#
# Same as checking variable availability in SAS
# before writing survival analysis code
# ============================================================

from pybioportal import clinical_attributes as ca

print("Checking available clinical attributes in this study...")
print("Looking specifically for survival variables...")

# Get all clinical attributes available in this study
attributes = ca.get_all_clinical_attributes_in_study(
    study_id=STUDY_ID
)

# Print all available attributes
print(f"\nAll available attributes:")
print(attributes[['clinicalAttributeId', 'displayName']])

# Specifically check for survival variables
# by searching for OS in the attribute IDs
survival_attrs = attributes[
    attributes['clinicalAttributeId'].str.contains(
        'OS|SURVIVAL|STATUS|MONTHS',
        case=False
    )
]

print(f"\nSurvival related attributes found:")
print(survival_attrs[['clinicalAttributeId', 'displayName']])

Checking available clinical attributes in this study...
Looking specifically for survival variables...

All available attributes:
          clinicalAttributeId                                   displayName
0   AGE_AT_SEQ_REPORTED_YEARS  Age at Which Sequencing was Reported (Years)
1                 AGE_CURRENT                           Patient Current Age
2                 CANCER_TYPE                                   Cancer Type
3        CANCER_TYPE_DETAILED                          Cancer Type Detailed
4                   ETHNICITY                            Ethnicity Category
5     FRACTION_GENOME_ALTERED                       Fraction Genome Altered
6                  GENE_PANEL                                    Gene Panel
7             METASTATIC_SITE                               Metastatic Site
8                   MSI_SCORE                                     MSI Score
9                    MSI_TYPE                                      MSI Type
10             MUTATION_COUNT     

In [24]:
# ============================================================
# CELL 10 - TMB Comparison: ESR1+ vs ESR1-
# ============================================================
# Tumor Mutation Burden (TMB) measures how many mutations
# exist per megabase of DNA in a tumor
# Higher TMB = more mutations = potentially more
# responsive to immunotherapy
#
# WHY THIS MATTERS:
# If ESR1+ patients have significantly different TMB
# it suggests a different underlying tumor biology
# This adds clinical context to our validation
# and is commonly reported in oncology studies
#
# STATISTICAL TEST:
# We use Mann-Whitney U test (not t-test)
# because TMB data is not normally distributed
# Mann-Whitney compares medians between two groups
# Same as Wilcoxon rank sum test in SAS:
# proc npar1way wilcoxon;
#   class ESR1_STATUS;
#   var TMB_NONSYNONYMOUS;
# run;
#
# INTERPRETATION:
# p < 0.05 → statistically significant difference
# p > 0.05 → no significant difference
# ============================================================

print("=" * 55)
print("TMB ANALYSIS: ESR1+ vs ESR1-")
print("=" * 55)

# ── PREPARE DATA ──────────────────────────────────────────
# Convert TMB to numeric
# errors='coerce' turns any non-numeric values to NaN
# so they get excluded from analysis automatically
merged_data['TMB_NUMERIC'] = pd.to_numeric(
    merged_data['TMB_NONSYNONYMOUS'],
    errors='coerce'
)

# Split TMB values by ESR1 status
# dropna() removes missing values before analysis
tmb_overall = merged_data['TMB_NUMERIC'].dropna()

tmb_esr1_pos = merged_data[
    merged_data['ESR1_STATUS'] == 'ESR1 Positive'
]['TMB_NUMERIC'].dropna()

tmb_esr1_neg = merged_data[
    merged_data['ESR1_STATUS'] == 'ESR1 Negative'
]['TMB_NUMERIC'].dropna()

# ── DESCRIPTIVE STATISTICS ────────────────────────────────
# Calculate median and IQR for each group
# IQR = Q3 - Q1 (interquartile range)
# Standard way to report non-normal continuous data

def get_median_iqr(data):
    """Returns median and IQR as formatted string"""
    median = data.median()
    q1 = data.quantile(0.25)
    q3 = data.quantile(0.75)
    return median, q1, q3

overall_med, overall_q1, overall_q3 = get_median_iqr(tmb_overall)
pos_med, pos_q1, pos_q3 = get_median_iqr(tmb_esr1_pos)
neg_med, neg_q1, neg_q3 = get_median_iqr(tmb_esr1_neg)

print(f"\nTMB Distribution:")
print(f"{'Group':<20} {'Median':>8} {'Q1':>8} {'Q3':>8} {'N':>6}")
print("-" * 55)
print(f"{'Overall':<20} {overall_med:>8.2f} {overall_q1:>8.2f} "
      f"{overall_q3:>8.2f} {len(tmb_overall):>6}")
print(f"{'ESR1 Positive':<20} {pos_med:>8.2f} {pos_q1:>8.2f} "
      f"{pos_q3:>8.2f} {len(tmb_esr1_pos):>6}")
print(f"{'ESR1 Negative':<20} {neg_med:>8.2f} {neg_q1:>8.2f} "
      f"{neg_q3:>8.2f} {len(tmb_esr1_neg):>6}")

# ── STATISTICAL TEST ──────────────────────────────────────
# Mann-Whitney U test compares distributions
# between two independent groups
# Does not assume normal distribution
# Perfect for TMB which is right-skewed
# scipy.stats.mannwhitneyu runs this test
statistic, p_value = sp.mannwhitneyu(
    tmb_esr1_pos,
    tmb_esr1_neg,
    alternative='two-sided'  # test for any difference
                              # not just one direction
)

# Round p-value for clean display
p_value_rounded = round(p_value, 4)

print(f"\n── Statistical Test ──")
print(f"Test: Mann-Whitney U (Wilcoxon Rank Sum)")
print(f"Statistic: {round(statistic, 2)}")
print(f"P-value: {p_value_rounded}")

# Interpret significance
if p_value < 0.05:
    print(f"\n✅ STATISTICALLY SIGNIFICANT (p < 0.05)")
    print(f"ESR1+ patients have significantly different")
    print(f"TMB compared to ESR1- patients")
else:
    print(f"\n➡️ NOT STATISTICALLY SIGNIFICANT (p > 0.05)")
    print(f"No significant TMB difference between")
    print(f"ESR1+ and ESR1- patients")

# ── STORE RESULTS FOR AGENT PIPELINE ─────────────────────
# Store TMB findings as dictionary
# for comparison and report agents
tmb_results = {
    "overall_median_tmb": round(overall_med, 2),
    "overall_iqr": f"{round(overall_q1, 2)}-{round(overall_q3, 2)}",
    "esr1_pos_median_tmb": round(pos_med, 2),
    "esr1_pos_iqr": f"{round(pos_q1, 2)}-{round(pos_q3, 2)}",
    "esr1_neg_median_tmb": round(neg_med, 2),
    "esr1_neg_iqr": f"{round(neg_q1, 2)}-{round(neg_q3, 2)}",
    "p_value": p_value_rounded,
    "significant": p_value < 0.05
}

print(f"\n✅ TMB results stored for agent pipeline!")
print(f"tmb_results: {tmb_results}")

TMB ANALYSIS: ESR1+ vs ESR1-

TMB Distribution:
Group                  Median       Q1       Q3      N
-------------------------------------------------------
Overall                  3.91     2.22     6.05   1116
ESR1 Positive            4.32     2.94     6.05    238
ESR1 Negative            3.91     2.22     6.05    878

── Statistical Test ──
Test: Mann-Whitney U (Wilcoxon Rank Sum)
Statistic: 115625.5
P-value: 0.0114

✅ STATISTICALLY SIGNIFICANT (p < 0.05)
ESR1+ patients have significantly different
TMB compared to ESR1- patients

✅ TMB results stored for agent pipeline!
tmb_results: {'overall_median_tmb': np.float64(3.91), 'overall_iqr': '2.22-6.05', 'esr1_pos_median_tmb': np.float64(4.32), 'esr1_pos_iqr': '2.94-6.05', 'esr1_neg_median_tmb': np.float64(3.91), 'esr1_neg_iqr': '2.22-6.05', 'p_value': np.float64(0.0114), 'significant': np.True_}


In [25]:
# ============================================================
# CELL 11 - TMB Visualization: ESR1+ vs ESR1-
# ============================================================
# Visual representation of our TMB analysis
# A boxplot shows median, IQR and outliers
# in one clean chart
#
# WHY A BOXPLOT:
# TMB data is not normally distributed
# Boxplot shows the full distribution
# without assuming normality
# Standard visualization in oncology papers
#
# WHAT THE BOXPLOT SHOWS:
# Box      → IQR (25th to 75th percentile)
# Line     → Median
# Whiskers → 1.5x IQR range
# Dots     → Outliers beyond whiskers
#
# THREE GROUPS:
# Overall population
# ESR1 Positive patients
# ESR1 Negative patients
# ============================================================

# Set up the figure
# figsize controls width and height in inches
fig, ax = plt.subplots(figsize=(10, 6))

# ── PREPARE DATA FOR PLOTTING ─────────────────────────────
# Create three separate arrays for each group
# matplotlib needs data as separate lists

overall_tmb = merged_data['TMB_NUMERIC'].dropna().tolist()
pos_tmb = merged_data[
    merged_data['ESR1_STATUS'] == 'ESR1 Positive'
]['TMB_NUMERIC'].dropna().tolist()
neg_tmb = merged_data[
    merged_data['ESR1_STATUS'] == 'ESR1 Negative'
]['TMB_NUMERIC'].dropna().tolist()

# ── CREATE BOXPLOT ────────────────────────────────────────
# patch_artist=True fills the boxes with color
# medianprops controls the median line appearance
bp = ax.boxplot(
    [overall_tmb, pos_tmb, neg_tmb],
    patch_artist=True,    # fill boxes with color
    labels=[
        f'Overall\n(N={len(overall_tmb)})',
        f'ESR1+\n(N={len(pos_tmb)})',
        f'ESR1-\n(N={len(neg_tmb)})'
    ],
    medianprops=dict(
        color='black',    # median line color
        linewidth=2       # median line thickness
    ),
    flierprops=dict(
        marker='o',       # outlier marker shape
        markersize=3,     # outlier marker size
        alpha=0.5         # outlier transparency
    )
)

# ── COLOR THE BOXES ───────────────────────────────────────
# Set different colors for each group
# alpha controls transparency (0=invisible, 1=solid)
colors = ['#AED6F1', '#E74C3C', '#2ECC71']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# ── ADD P-VALUE ANNOTATION ────────────────────────────────
# Show the statistical test result on the plot
# This is standard in oncology publications
ax.annotate(
    f'p = {p_value_rounded}\n(Mann-Whitney U)',
    xy=(0.98, 0.95),           # position in plot
    xycoords='axes fraction',  # relative to plot area
    ha='right',                # right aligned
    va='top',                  # top aligned
    fontsize=11,
    bbox=dict(
        boxstyle='round',      # rounded box around text
        facecolor='wheat',     # background color
        alpha=0.5              # transparency
    )
)

# ── ADD MEDIAN LABELS ON BOXES ────────────────────────────
# Show exact median values on each box
# Makes the chart easier to read
medians = [
    round(pd.Series(overall_tmb).median(), 2),
    round(pd.Series(pos_tmb).median(), 2),
    round(pd.Series(neg_tmb).median(), 2)
]

for i, median in enumerate(medians):
    ax.text(
        i + 1,          # x position (1 indexed)
        median + 0.2,   # slightly above median line
        f'{median}',    # the median value as text
        ha='center',    # centered horizontally
        va='bottom',    # above the point
        fontsize=10,
        fontweight='bold'
    )

# ── FORMATTING ────────────────────────────────────────────
# Add title and axis labels
ax.set_title(
    'Tumor Mutation Burden (TMB) by ESR1 Mutation Status\n'
    'MSK Metastatic Breast Cancer (Cancer Discovery 2022)',
    fontsize=13,
    fontweight='bold',
    pad=15
)
ax.set_ylabel('TMB (Nonsynonymous mutations/Mb)', fontsize=12)
ax.set_xlabel('ESR1 Mutation Status', fontsize=12)

# Set y axis limit
# 40 captures most data without being distorted by outliers
ax.set_ylim(0, 40)

# Add light grid for easier reading
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)  # grid behind the boxes

# ── SAVE THE PLOT ─────────────────────────────────────────
# Save as PNG file in current folder
# dpi=150 gives good quality without huge file size
# bbox_inches='tight' prevents labels being cut off
plot_filename = "tmb_esr1_comparison.png"
plt.savefig(
    plot_filename,
    dpi=150,
    bbox_inches='tight',
    facecolor='white'   # white background
)

print(f"✅ Plot saved as: {plot_filename}")
plt.show()
print("\nPlot interpretation:")
print(f"ESR1+ median TMB: {medians[1]} vs ESR1- median TMB: {medians[2]}")
print(f"Difference is statistically significant (p={p_value_rounded})")

✅ Plot saved as: tmb_esr1_comparison.png

Plot interpretation:
ESR1+ median TMB: 4.32 vs ESR1- median TMB: 3.91
Difference is statistically significant (p=0.0114)
